# ResNet-50 classifier: train and score

Trains a ResNet-50 on the shared pancreas crops, picks how slice scores become a patient score using the validation patients, and scores the test patients once. Needs crops.zip and classification.zip in the PanVision folder on Google Drive, and a T4 GPU (Runtime → Change runtime type).

## 1. Connect to Google Drive
Gives Colab access to the zips and lets it save the trained model back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Unpack the crops and the code
Copies everything into a fresh working folder on Colab. This deletes the old working folder on Colab only, nothing on Drive.

In [ ]:
!rm -rf /content/work
!mkdir -p /content/work/outputs /content/work/src
!unzip -q /content/drive/MyDrive/PanVision/crops.zip -d /content/work/outputs
!unzip -q /content/drive/MyDrive/PanVision/classification.zip -d /content/work/src

## 3. Check that the files are there
You should see the code files (including `compare_aggregations.py` and `evaluate_model.py`) and a list of `.npz` crop files.

In [ ]:
!ls /content/work/src/classification
!ls /content/work/outputs/crops | head -3

compare_aggregations.py  evaluate_model.py  __pycache__      train_resnet.py
crops_summary.py	 evaluation.py	    README.md	     view_crops.py
dataset.py		 make_crops.py	    resnet_model.py
100000_00001.npz
100001_00001.npz
100002_00001.npz


## 4. Train ResNet-50
Trains for 8 epochs and keeps the model with the best validation AUC. Takes about 46 minutes.

**Warning:** this saves to `classifier.pt` on Drive and overwrites the existing model. Skip this cell if the trained model already exists.

In [ ]:
%cd /content/work
!python src/classification/train_resnet.py --crops outputs/crops --epochs 8 --out /content/drive/MyDrive/PanVision/classifier.pt

/content/work
Training on: cuda
Train: 26483 pictures from 1551 patients (469 PDAC). PDAC pictures weighted x5.4.
Validation: 6818 pictures from 334 patients (101 PDAC).
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 185MB/s]
Epoch 1/8  train loss 0.4055  |  validation: AUC 0.885 | sensitivity 0.59 | specificity 0.91 | accuracy 0.82 (threshold 0.50, 334 patients, 101 PDAC)
   saved /content/drive/MyDrive/PanVision/classifier.pt (best so far, suggested threshold 0.20)
Epoch 2/8  train loss 0.2236  |  validation: AUC 0.908 | sensitivity 0.27 | specificity 1.00 | accuracy 0.78 (threshold 0.50, 334 patients, 101 PDAC)
   saved /content/drive/MyDrive/PanVision/classifier.pt (best so far, suggested threshold 0.11)
Epoch 3/8  train loss 0.1462  |  validation: AUC 0.894 | sensitivity 0.39 | specificity 0.98 | accuracy 0.80 (threshold 0.50, 334 patients, 101 PDAC)
Epoch 4/8  trai

## 5. Choose how slice scores become a patient score
Tries eight rules (average, maximum, top-k slices, ...) on the **validation** patients only. All eight land within the noise (AUC 0.906 to 0.925), so the simplest rule, the plain **mean**, is used. The test patients are not touched here.

In [ ]:
%cd /content/work
!nvidia-smi --query-gpu=name --format=csv
!python src/classification/compare_aggregations.py --model /content/drive/MyDrive/PanVision/classifier.pt --crops outputs/crops

/content/work
name
Tesla T4
Saved slice scores to outputs/val_probs.npz

Validation patients: 334 (101 PDAC)
rule          AUC  cut-off  sensitivity  specificity
mean        0.920    0.125         0.87         0.84
max         0.906    0.945         0.81         0.85
topk:3      0.922    0.534         0.91         0.82
topk:5      0.922    0.345         0.91         0.81
topk:10     0.925    0.229         0.89         0.84
frac:0.1    0.920    0.616         0.89         0.82
frac:0.2    0.920    0.427         0.89         0.82
frac:0.5    0.922    0.247         0.86         0.85

Pick ONE rule and keep it. Differences of about 0.01 in AUC are just noise.


## 6. Score the test patients (run once)
The cut-off (0.13) is chosen on the validation patients and applied unchanged to the test patients. Run this once, at the very end. Re-running it and changing settings would make the test result unreliable.

In [ ]:
!python src/classification/evaluate_model.py --model /content/drive/MyDrive/PanVision/classifier.pt --crops outputs/crops --split test --aggregation mean

Model from epoch 8, validation AUC 0.9203671440105384; scoring rule: mean

VALIDATION (cut-off 0.125 chosen here): AUC 0.920 | sensitivity 0.87 | specificity 0.84 | accuracy 0.85 (threshold 0.13, 334 patients, 101 PDAC)

TEST: AUC 0.882 | sensitivity 0.84 | specificity 0.81 | accuracy 0.82 (threshold 0.13, 334 patients, 101 PDAC)
TEST AUC 95% range from resampling the patients: 0.840 to 0.920
(a wide range means the set is too small to separate close models)
Saved outputs/test_scores.csv


## 7. Save the per-patient test scores to Drive
One row per test patient: id, true label, model score.

In [ ]:
!cp outputs/test_scores.csv /content/drive/MyDrive/PanVision/test_scores_resnet50.csv